# Notebook 02 — Projective Geometry

**Vision & 3D Mapping Workshop** | Block 1: Mathematical Foundations

---

## Why This Matters

Projective geometry is the **mathematical language of computer vision** — and the backbone of
perception in autonomous systems. Every time a camera captures an image, it performs a
projective transformation: parallel lines converge, shapes distort, and the 3D world collapses
onto a 2D sensor. A self-driving car uses homographies to warp a front camera into a bird's-eye
view for lane detection; a drone stitches panoramas via projective transforms to build terrain
maps; RANSAC filters outlier feature matches in every real-time visual odometry pipeline.
Understanding projective geometry lets you *undo* that collapse and reason about 3D structure
from 2D images.

### What You'll Learn

1. **Homogeneous coordinates** — the algebraic trick that makes projective geometry linear
2. **2D transformation hierarchy** — from rigid motions to full homographies
3. **Homography estimation (DLT)** — the complete SVD derivation with Hartley normalization
4. **RANSAC** — robust estimation in the presence of outliers
5. **Image warping** — inverse mapping and bilinear interpolation
6. **Vanishing points & lines** — where parallel lines meet
7. **Cross-ratio invariance** — the fundamental projective invariant (with proof)
8. **Single-view metrology** — measuring heights from one photograph

### Prerequisites
- Linear algebra (matrix multiplication, SVD, eigenvalues)
- Notebook 01 (image basics, convolution)

### References
- Hartley & Zisserman, *Multiple View Geometry in Computer Vision*, 2nd ed.
- Szeliski, *Computer Vision: Algorithms and Applications*, 2nd ed., Ch. 2 & 8

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.collections import LineCollection
import cv2

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["image.cmap"] = "gray"
np.set_printoptions(precision=6, suppress=True)

---
## 1. Homogeneous Coordinates

### 1.1 Motivation: Why Add a 1?

In Euclidean geometry, a 2D point is a pair $(x, y) \in \mathbb{R}^2$. But translation
is not a linear operation — it can't be expressed as a matrix multiplication:

$$
\begin{pmatrix} x' \\ y' \end{pmatrix} = 
\begin{pmatrix} a & b \\ c & d \end{pmatrix}
\begin{pmatrix} x \\ y \end{pmatrix} +
\begin{pmatrix} t_x \\ t_y \end{pmatrix}
$$

That additive $\mathbf{t}$ breaks linearity. The fix: embed $\mathbb{R}^2$ into
$\mathbb{P}^2$ (the projective plane) by **appending a 1**:

$$
\mathbf{x} = \begin{pmatrix} x \\ y \end{pmatrix}
\;\longrightarrow\;
\tilde{\mathbf{x}} = \begin{pmatrix} x \\ y \\ 1 \end{pmatrix}
$$

Now translation becomes a matrix multiplication:

$$
\begin{pmatrix} x + t_x \\ y + t_y \\ 1 \end{pmatrix} =
\begin{pmatrix} 1 & 0 & t_x \\ 0 & 1 & t_y \\ 0 & 0 & 1 \end{pmatrix}
\begin{pmatrix} x \\ y \\ 1 \end{pmatrix}
$$

### 1.2 The Projective Plane $\mathbb{P}^2$

A point in $\mathbb{P}^2$ is an **equivalence class** of 3-vectors:

$$
\tilde{\mathbf{x}} = \begin{pmatrix} x \\ y \\ w \end{pmatrix}
\sim \lambda \begin{pmatrix} x \\ y \\ w \end{pmatrix}
\quad \text{for any } \lambda \neq 0
$$

Two homogeneous vectors represent the **same point** if they differ by a non-zero scalar.
When $w \neq 0$, the Euclidean coordinates are recovered by dividing:

$$
\begin{pmatrix} x \\ y \\ w \end{pmatrix}
\;\longrightarrow\;
\begin{pmatrix} x/w \\ y/w \end{pmatrix}
$$

### 1.3 Points at Infinity

When $w = 0$, the point $(x, y, 0)$ cannot be de-homogenized — it corresponds to
a **point at infinity** (an ideal point), representing a *direction* rather than a location:

$$
\tilde{\mathbf{x}}_\infty = \begin{pmatrix} x \\ y \\ 0 \end{pmatrix}
\;\longleftrightarrow\;
\text{the direction } (x, y)
$$

The set of all ideal points forms the **line at infinity** $\ell_\infty = (0, 0, 1)$.
Points at infinity are where parallel lines meet — we'll see this concretely in Section 6.

### 1.4 Lines as 3-Vectors

A line in 2D is defined by $ax + by + c = 0$. We represent it as the 3-vector
$\mathbf{l} = (a, b, c)^T$. The condition for a point $\tilde{\mathbf{x}}$ to lie
on line $\mathbf{l}$ is:

$$
\mathbf{l}^T \tilde{\mathbf{x}} = 0
\quad \Longleftrightarrow \quad
ax + by + cw = 0
$$

This is a **symmetric** relation — we can equally say that the point lies on the line,
or the line passes through the point.

### 1.5 Line Through Two Points

**Claim**: The line through points $\tilde{\mathbf{p}}_1$ and $\tilde{\mathbf{p}}_2$ is:

$$
\mathbf{l} = \tilde{\mathbf{p}}_1 \times \tilde{\mathbf{p}}_2
$$

**Proof**: We need $\mathbf{l}^T \tilde{\mathbf{p}}_1 = 0$ and $\mathbf{l}^T \tilde{\mathbf{p}}_2 = 0$.
The cross product $\mathbf{a} \times \mathbf{b}$ is orthogonal to both $\mathbf{a}$ and $\mathbf{b}$:

$$
(\tilde{\mathbf{p}}_1 \times \tilde{\mathbf{p}}_2)^T \tilde{\mathbf{p}}_1
= \tilde{\mathbf{p}}_1 \cdot (\tilde{\mathbf{p}}_1 \times \tilde{\mathbf{p}}_2)
= 0
$$

by the scalar triple product identity $\mathbf{a} \cdot (\mathbf{a} \times \mathbf{b}) = 0$.
The same holds for $\tilde{\mathbf{p}}_2$. $\square$

### 1.6 Intersection of Two Lines

**Claim**: The intersection of lines $\mathbf{l}_1$ and $\mathbf{l}_2$ is:

$$
\tilde{\mathbf{x}} = \mathbf{l}_1 \times \mathbf{l}_2
$$

**Proof**: By the same argument — $\tilde{\mathbf{x}}$ must satisfy both $\mathbf{l}_1^T \tilde{\mathbf{x}} = 0$
and $\mathbf{l}_2^T \tilde{\mathbf{x}} = 0$, and the cross product is orthogonal to both inputs.

**Key insight**: If $\mathbf{l}_1$ and $\mathbf{l}_2$ are parallel, the third component of
$\mathbf{l}_1 \times \mathbf{l}_2$ is zero — the intersection is a point at infinity. Parallel
lines meet at infinity! This is exactly how projective geometry unifies Euclidean geometry.

### 1.7 Duality — Formal Statement and Proof

Notice the beautiful symmetry: both operations use the **same formula** (cross product).

| Operation | Formula |
|---|---|
| Line through 2 points | $\mathbf{l} = \mathbf{p}_1 \times \mathbf{p}_2$ |
| Intersection of 2 lines | $\mathbf{p} = \mathbf{l}_1 \times \mathbf{l}_2$ |
| Point on line | $\mathbf{l}^T \mathbf{p} = 0$ |
| Line through point | $\mathbf{p}^T \mathbf{l} = 0$ |

**Theorem (Principle of Duality in $\mathbb{P}^2$).** *Let $\mathcal{S}$ be any theorem
about points and lines in the projective plane $\mathbb{P}^2$ expressed solely in terms
of incidence and cross products. Then the dual statement $\mathcal{S}^*$, obtained by
interchanging "point" $\leftrightarrow$ "line", is also a theorem.*

**Proof.** In $\mathbb{P}^2$, both points and lines are represented as equivalence classes
of non-zero 3-vectors: $\tilde{\mathbf{x}} \in (\mathbb{R}^3 \setminus \{\mathbf{0}\})/{\sim}$.
The fundamental incidence relation is:

$$
\mathbf{l}^T \mathbf{p} = 0
$$

This relation is **symmetric** in its operands: $\mathbf{l}^T \mathbf{p} = \mathbf{p}^T \mathbf{l}$.
Therefore we can define a duality map $\delta$ that sends each point $\mathbf{p}$ to the line
represented by the same vector, and each line $\mathbf{l}$ to the point represented by the
same vector.

**$\delta$ preserves incidence:** "line $\mathbf{l}$ passes through point $\mathbf{p}$" is
$\mathbf{l}^T \mathbf{p} = 0$, which is equivalent to $\mathbf{p}^T \mathbf{l} = 0$, i.e.,
"point $\delta(\mathbf{l})$ lies on line $\delta(\mathbf{p})$" in the dual.

**$\delta$ preserves join/meet:** The cross product $\mathbf{a} \times \mathbf{b}$ computes
both "line through two points" and "intersection of two lines." Under $\delta$:

$$
\delta(\mathbf{p}_1 \times \mathbf{p}_2) = \delta(\mathbf{l})
\quad \longleftrightarrow \quad
\delta(\mathbf{l}_1) \times \delta(\mathbf{l}_2) = \delta(\mathbf{p})
$$

Since every projective-geometric statement about points and lines can be expressed using
only incidence ($\mathbf{l}^T \mathbf{p} = 0$) and the cross product, applying $\delta$
transforms any valid theorem into another valid theorem. $\square$

**Example of duality in action:** "Two distinct points determine a unique line"
dualizes to "Two distinct lines determine a unique point" (their intersection).
The first is $\mathbf{l} = \mathbf{p}_1 \times \mathbf{p}_2$; the second is
$\mathbf{p} = \mathbf{l}_1 \times \mathbf{l}_2$. Both follow from the same
algebraic operation.

In [ ]:
def to_homogeneous(points):
    """Convert Euclidean points to homogeneous: (N, D) -> (N, D+1)."""
    points = np.asarray(points, dtype=np.float64)
    if points.ndim == 1:
        return np.append(points, 1.0)
    ones = np.ones((points.shape[0], 1), dtype=np.float64)
    return np.hstack([points, ones])


def from_homogeneous(points):
    """Convert homogeneous to Euclidean: divide by last coordinate."""
    points = np.asarray(points, dtype=np.float64)
    if points.ndim == 1:
        return points[:-1] / points[-1]
    return points[:, :-1] / points[:, -1:]


# --- Demonstrate homogeneous coordinates ---
p = np.array([3.0, 4.0])
p_h = to_homogeneous(p)
print(f"Euclidean point:    {p}")
print(f"Homogeneous:        {p_h}")
print(f"Scaled (x2):        {2 * p_h}  (same projective point)")
print(f"Back to Euclidean:  {from_homogeneous(2 * p_h)}")

p_inf = np.array([1.0, 2.0, 0.0])
print(f"\nPoint at infinity:  {p_inf}  (direction (1,2), w=0)")
print(f"De-homogenize:      {from_homogeneous(p_inf)}  (goes to infinity!)")

In [ ]:
# --- Lines and their operations ---

p1 = to_homogeneous(np.array([1.0, 1.0]))
p2 = to_homogeneous(np.array([4.0, 3.0]))

line = np.cross(p1, p2)
print(f"Points: p1={p1}, p2={p2}")
print(f"Line through p1, p2:  l = {line}")
print(f"  (equation: {line[0]:.3f}x + {line[1]:.3f}y + {line[2]:.3f} = 0)")
print(f"  Verify: l·p1 = {np.dot(line, p1):.10f}")
print(f"  Verify: l·p2 = {np.dot(line, p2):.10f}")

# Two lines intersecting
l1 = np.array([1.0, -1.0, 0.0])    # y = x
l2 = np.array([1.0, 1.0, -4.0])    # x + y = 4
intersection = np.cross(l1, l2)
intersection_e = from_homogeneous(intersection)
print(f"\nLine l1: x - y = 0  (y = x)")
print(f"Line l2: x + y = 4")
print(f"Intersection: {intersection_e}  (should be (2, 2))")

# Parallel lines meet at infinity
l3 = np.array([0.0, 1.0, -1.0])    # y = 1
l4 = np.array([0.0, 1.0, -3.0])    # y = 3
p_inf = np.cross(l3, l4)
print(f"\nParallel lines: y=1 and y=3")
print(f"Intersection: {p_inf}  (w=0, a point at infinity in direction x!)")

In [ ]:
# --- Visualization: lines, intersection, and duality ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
ax.set_title("Lines and Intersections in $\\mathbb{P}^2$", fontsize=13)

points = np.array([[1, 1], [4, 3], [0, 2], [5, 1]])
for i, (label, pt) in enumerate(zip(["$p_1$", "$p_2$", "$p_3$", "$p_4$"], points)):
    ax.plot(*pt, "o", markersize=8)
    ax.annotate(label, pt, textcoords="offset points", xytext=(8, 8), fontsize=12)

def draw_line_segment(ax, l, xlim, color, label=None):
    """Draw the line l = (a,b,c) across the given x-limits."""
    a, b, c = l
    if abs(b) > 1e-10:
        x = np.linspace(*xlim, 100)
        y = -(a * x + c) / b
        ax.plot(x, y, color=color, linewidth=1.5, label=label)
    else:
        yy = np.linspace(-1, 6, 100)
        xx = np.full_like(yy, -c / a)
        ax.plot(xx, yy, color=color, linewidth=1.5, label=label)

p1_h, p2_h = to_homogeneous(points[0]), to_homogeneous(points[1])
p3_h, p4_h = to_homogeneous(points[2]), to_homogeneous(points[3])
l_12 = np.cross(p1_h, p2_h)
l_34 = np.cross(p3_h, p4_h)
draw_line_segment(ax, l_12, (-1, 6), "C0", "$\\ell_{12} = p_1 \\times p_2$")
draw_line_segment(ax, l_34, (-1, 6), "C1", "$\\ell_{34} = p_3 \\times p_4$")

x_int = np.cross(l_12, l_34)
x_int_e = from_homogeneous(x_int)
ax.plot(*x_int_e, "r*", markersize=15, zorder=5)
ax.annotate(f"$\\ell_{{12}} \\times \\ell_{{34}}$ = ({x_int_e[0]:.2f}, {x_int_e[1]:.2f})",
            x_int_e, textcoords="offset points", xytext=(10, -15), fontsize=11, color="red")

ax.set_xlim(-1, 6)
ax.set_ylim(-1, 5)
ax.set_aspect("equal")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Right panel: parallel lines meeting at infinity
ax = axes[1]
ax.set_title("Parallel Lines Meet at Infinity", fontsize=13)
for y_val, col in zip([0.5, 1.5, 2.5, 3.5], ["C0", "C1", "C2", "C3"]):
    ax.axhline(y=y_val, color=col, linewidth=2, alpha=0.7)
    ax.annotate(f"y = {y_val}", (0.1, y_val), textcoords="offset points",
                xytext=(0, 8), fontsize=10, color=col)

ax.annotate("All meet at\n$\\mathbf{p}_\\infty = (1, 0, 0)$",
            xy=(5, 2.0), fontsize=13, color="red",
            bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow", ec="red"))
ax.arrow(5, 1.5, 1.5, 0, head_width=0.15, head_length=0.1, fc="red", ec="red")
ax.set_xlim(-0.5, 7.5)
ax.set_ylim(-0.5, 4.5)
ax.set_aspect("equal")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 1b. Conics in the Projective Plane

### 1b.1 Definition

A **conic** in $\mathbb{P}^2$ is the set of points $\tilde{\mathbf{x}}$ satisfying:

$$
\tilde{\mathbf{x}}^T C \tilde{\mathbf{x}} = 0
$$

where $C$ is a $3 \times 3$ **symmetric** matrix. Since $\tilde{\mathbf{x}}$ is homogeneous,
$C$ is defined up to scale, so a conic has $\frac{3 \cdot 4}{2} - 1 = 5$ degrees of freedom.
**Five points in general position uniquely determine a conic.**

### 1b.2 Point Conic and Dual (Line) Conic

By projective duality (Section 1.7), every point conic has a **dual conic** (or **line conic**):
the set of lines $\mathbf{l}$ tangent to the conic. If $C$ is non-singular, the dual conic is:

$$
\mathbf{l}^T C^{-1} \mathbf{l} = 0
$$

**Derivation.** A line $\mathbf{l}$ is tangent to the conic $C$ at point $\mathbf{x}$ if and
only if $\mathbf{l} = C\mathbf{x}$ (the gradient of $\tilde{\mathbf{x}}^T C\tilde{\mathbf{x}}$
at $\mathbf{x}$). Substituting $\mathbf{x} = C^{-1}\mathbf{l}$:

$$
\mathbf{l}^T C^{-1} \mathbf{l} = (C \mathbf{x})^T C^{-1} (C \mathbf{x})
= \mathbf{x}^T C C^{-1} C \mathbf{x} = \mathbf{x}^T C \mathbf{x} = 0 \quad \square
$$

### 1b.3 Classification by Rank

| $\operatorname{rank}(C)$ | $\det(C)$ | Geometric Type |
|:---|:---|:---|
| 3 | $\neq 0$ | **Non-degenerate conic** (ellipse, parabola, hyperbola in affine view) |
| 2 | $= 0$ | **Two distinct lines** (degenerate — the conic "splits") |
| 1 | $= 0$ | **A repeated (double) line** |

### 1b.4 Projective Equivalence — All Non-Degenerate Conics Are the Same

**Theorem.** Under projective transformations, all non-degenerate conics are equivalent.

**Proof.** Under a point transformation $\tilde{\mathbf{x}}' = H\tilde{\mathbf{x}}$, the
conic $C$ transforms as:

$$
C' = H^{-T} C \, H^{-1}
$$

*Derivation:* If $\tilde{\mathbf{x}}^T C \tilde{\mathbf{x}} = 0$ and
$\tilde{\mathbf{x}} = H^{-1}\tilde{\mathbf{x}}'$, then:

$$
(H^{-1}\tilde{\mathbf{x}}')^T C (H^{-1}\tilde{\mathbf{x}}')
= \tilde{\mathbf{x}}'^{\,T} \underbrace{H^{-T} C H^{-1}}_{C'} \tilde{\mathbf{x}}' = 0
$$

Now, $C$ is a real symmetric matrix, so by the **spectral theorem** it has real eigenvalues.
For a non-degenerate conic, $\operatorname{rank}(C) = 3$ and the eigenvalues are all
non-zero. By **Sylvester's law of inertia**, the only projective invariant of a real
symmetric matrix under congruence $C' = H^{-T}CH^{-1}$ is its **signature** — the
number of positive and negative eigenvalues.

Every non-degenerate conic in $\mathbb{P}^2$ has exactly **2 eigenvalues of one sign and
1 of the other** (signature $(2, 1)$), because if all three had the same sign, only
$\tilde{\mathbf{x}} = \mathbf{0}$ would satisfy $\tilde{\mathbf{x}}^TC\tilde{\mathbf{x}} = 0$
(no real solutions in $\mathbb{P}^2$).

Therefore any non-degenerate conic can be transformed to the canonical form:

$$
C_0 = \operatorname{diag}(1, 1, -1) \qquad \Longleftrightarrow \qquad x_1^2 + x_2^2 - x_3^2 = 0
$$

which is a circle in affine coordinates. In projective geometry, there is no distinction
between ellipse, parabola, and hyperbola — they are all projectively equivalent. The
distinction only appears when we fix the line at infinity $\ell_\infty = (0, 0, 1)^T$:

| Affine Type | Intersections with $\ell_\infty$ | Discriminant |
|:---|:---|:---|
| **Ellipse** | 0 real intersections | $B^2 - 4AC < 0$ |
| **Parabola** | 1 real intersection (tangent) | $B^2 - 4AC = 0$ |
| **Hyperbola** | 2 real intersections | $B^2 - 4AC > 0$ |

where $Ax^2 + Bxy + Cy^2 + Dx + Ey + F = 0$ is the affine conic equation. $\square$

---
## 2. 2D Transformation Hierarchy

Projective geometry organizes 2D transformations into a strict hierarchy. Each level
preserves fewer geometric properties but has more degrees of freedom (DOF).

### 2.1 Isometry (Euclidean) — 3 DOF

An isometry preserves **distances** and **angles**. It consists of a rotation $R$ and
translation $\mathbf{t}$:

$$
H_E = \begin{pmatrix} R & \mathbf{t} \\ \mathbf{0}^T & 1 \end{pmatrix}
= \begin{pmatrix} \cos\theta & -\sin\theta & t_x \\ \sin\theta & \cos\theta & t_y \\ 0 & 0 & 1 \end{pmatrix}
$$

**DOF count**: $\theta$ (1) + $(t_x, t_y)$ (2) = **3 DOF**.

**Preserves**: distances, angles, area, parallelism, everything geometric.

### 2.2 Similarity — 4 DOF

A similarity adds **uniform scaling** $s > 0$:

$$
H_S = \begin{pmatrix} sR & \mathbf{t} \\ \mathbf{0}^T & 1 \end{pmatrix}
= \begin{pmatrix} s\cos\theta & -s\sin\theta & t_x \\ s\sin\theta & s\cos\theta & t_y \\ 0 & 0 & 1 \end{pmatrix}
$$

**DOF**: $s$ (1) + $\theta$ (1) + $(t_x, t_y)$ (2) = **4 DOF**.

**Preserves**: angles, ratios of distances.
**Breaks**: absolute distances (scaled by $s$).

### 2.3 Affine — 6 DOF

An affine transformation uses a general $2 \times 2$ matrix $A$ (not necessarily a scaled rotation):

$$
H_A = \begin{pmatrix} A & \mathbf{t} \\ \mathbf{0}^T & 1 \end{pmatrix}
= \begin{pmatrix} a_{11} & a_{12} & t_x \\ a_{21} & a_{22} & t_y \\ 0 & 0 & 1 \end{pmatrix}
$$

**Derivation of DOF**: $A$ has 4 entries + $(t_x, t_y)$ gives 2 = **6 DOF**.

**Preserves**: parallelism (parallel lines stay parallel), ratios of areas, ratios of
lengths along any line.

**Breaks**: angles (rectangles become parallelograms).

### 2.4 Projective (Homography) — 8 DOF

The most general linear transformation of $\mathbb{P}^2$:

$$
H = \begin{pmatrix} h_{11} & h_{12} & h_{13} \\ h_{21} & h_{22} & h_{23} \\ h_{31} & h_{32} & h_{33} \end{pmatrix}
$$

**DOF count**: $H$ has 9 entries, but it's defined only up to scale ($H \sim \lambda H$),
so **8 DOF**.

**Preserves**: collinearity (points on a line stay on a line), cross-ratio.

**Breaks**: parallelism (parallel lines can converge), angles, distances.

### Hierarchy Summary

$$
\text{Isometry} \subset \text{Similarity} \subset \text{Affine} \subset \text{Projective}
$$

| Transformation | DOF | Matrix form | Preserves |
|---|---|---|---|
| Isometry | 3 | $\begin{pmatrix} R & \mathbf{t} \\ \mathbf{0}^T & 1 \end{pmatrix}$ | Distances, angles |
| Similarity | 4 | $\begin{pmatrix} sR & \mathbf{t} \\ \mathbf{0}^T & 1 \end{pmatrix}$ | Angles, ratios |
| Affine | 6 | $\begin{pmatrix} A & \mathbf{t} \\ \mathbf{0}^T & 1 \end{pmatrix}$ | Parallelism |
| Projective | 8 | $\begin{pmatrix} A & \mathbf{t} \\ \mathbf{v}^T & v \end{pmatrix}$ | Collinearity, cross-ratio |

In [ ]:
def make_checkerboard(rows=4, cols=4, square_size=50):
    """Generate a checkerboard pattern and its corner points."""
    h, w = rows * square_size, cols * square_size
    img = np.zeros((h, w), dtype=np.uint8)
    for r in range(rows):
        for c in range(cols):
            if (r + c) % 2 == 0:
                y0, y1 = r * square_size, (r + 1) * square_size
                x0, x1 = c * square_size, (c + 1) * square_size
                img[y0:y1, x0:x1] = 255

    corners = []
    for r in range(rows + 1):
        for c in range(cols + 1):
            corners.append([c * square_size, r * square_size])
    return img, np.array(corners, dtype=np.float64)


def apply_homography_to_points(H, pts):
    """Apply H to Euclidean 2D points, return Euclidean result."""
    pts_h = to_homogeneous(pts)
    result_h = (H @ pts_h.T).T
    return from_homogeneous(result_h)


checkerboard, cb_corners = make_checkerboard()

theta = np.radians(25)
tx, ty = 30, 15

# Isometry: rotation + translation
H_iso = np.array([
    [np.cos(theta), -np.sin(theta), tx],
    [np.sin(theta),  np.cos(theta), ty],
    [0, 0, 1]
])

# Similarity: scale + rotation + translation
s = 0.7
H_sim = np.array([
    [s * np.cos(theta), -s * np.sin(theta), tx],
    [s * np.sin(theta),  s * np.cos(theta), ty],
    [0, 0, 1]
])

# Affine: non-uniform scale + shear + translation
H_aff = np.array([
    [0.8, 0.3, tx],
    [0.1, 1.1, ty],
    [0, 0, 1]
])

# Projective: full homography with perspective
H_proj = np.array([
    [0.9, 0.2, tx],
    [0.1, 1.0, ty],
    [0.0005, 0.001, 1]
])

transforms = [
    ("Original", np.eye(3), "Distances, angles,\nparallelism, everything"),
    ("Isometry (3 DOF)", H_iso, "Preserves: distances, angles\nBreaks: position"),
    ("Similarity (4 DOF)", H_sim, "Preserves: angles, ratios\nBreaks: distances"),
    ("Affine (6 DOF)", H_aff, "Preserves: parallelism\nBreaks: angles"),
    ("Projective (8 DOF)", H_proj, "Preserves: collinearity\nBreaks: parallelism"),
]

fig, axes = plt.subplots(1, 5, figsize=(22, 4.5))
for ax, (title, H, desc) in zip(axes, transforms):
    pts_t = apply_homography_to_points(H, cb_corners)

    rows, cols = 5, 5
    for r in range(rows):
        row_pts = pts_t[r * cols:(r + 1) * cols]
        ax.plot(row_pts[:, 0], row_pts[:, 1], "b-", linewidth=0.8)
    for c in range(cols):
        col_pts = pts_t[c::cols]
        ax.plot(col_pts[:, 0], col_pts[:, 1], "b-", linewidth=0.8)

    ax.plot(pts_t[:, 0], pts_t[:, 1], "r.", markersize=3)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.text(0.5, -0.12, desc, transform=ax.transAxes, ha="center", fontsize=8,
            style="italic", va="top")
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.2)

plt.suptitle("2D Transformation Hierarchy — Applied to a Checkerboard",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

---
## 3. Homography Estimation — The DLT Algorithm with Full SVD Derivation

This is the **most important derivation** in this notebook. Given $N \ge 4$ correspondences
$\mathbf{x}_i \leftrightarrow \mathbf{x}'_i$, we want the $3 \times 3$ homography $H$ such that:

$$
\mathbf{x}'_i \sim H \mathbf{x}_i
$$

(the $\sim$ means "equal up to scale").

### 3.1 From $\mathbf{x}' \sim H\mathbf{x}$ to $\mathbf{x}' \times H\mathbf{x} = \mathbf{0}$

The relation $\mathbf{x}' \sim H\mathbf{x}$ means there exists a scalar $\lambda$ such that
$\mathbf{x}' = \lambda H\mathbf{x}$. Taking the cross product of both sides with $\mathbf{x}'$:

$$
\mathbf{x}' \times \mathbf{x}' = \lambda (\mathbf{x}' \times H\mathbf{x})
$$

Since $\mathbf{x}' \times \mathbf{x}' = \mathbf{0}$ always, and $\lambda \neq 0$:

$$
\boxed{\mathbf{x}' \times H\mathbf{x} = \mathbf{0}}
$$

### 3.2 Expanding the Cross Product

Write $H$ row by row: $H = \begin{pmatrix} \mathbf{h}_1^T \\ \mathbf{h}_2^T \\ \mathbf{h}_3^T \end{pmatrix}$,
so $H\mathbf{x} = \begin{pmatrix} \mathbf{h}_1^T \mathbf{x} \\ \mathbf{h}_2^T \mathbf{x} \\ \mathbf{h}_3^T \mathbf{x} \end{pmatrix}$
and $\mathbf{x}' = (x', y', w')^T$.

The cross product $\mathbf{x}' \times H\mathbf{x}$ gives three equations:

$$
\begin{aligned}
y' \mathbf{h}_3^T \mathbf{x} - w' \mathbf{h}_2^T \mathbf{x} &= 0 \\
w' \mathbf{h}_1^T \mathbf{x} - x' \mathbf{h}_3^T \mathbf{x} &= 0 \\
x' \mathbf{h}_2^T \mathbf{x} - y' \mathbf{h}_1^T \mathbf{x} &= 0
\end{aligned}
$$

The third equation is a linear combination of the first two (multiply eq.1 by $x'$, eq.2 by $y'$,
and add). So only **2 independent equations** per point.

### 3.3 Vectorizing into $A\mathbf{h} = \mathbf{0}$

Let $\mathbf{h} = \text{vec}(H) = (h_{11}, h_{12}, h_{13}, h_{21}, \ldots, h_{33})^T$ be
the 9-vector of $H$'s entries. With $w' = 1$ (normalized homogeneous coordinates), the
two equations for correspondence $i$ become:

$$
\begin{aligned}
\begin{pmatrix}
\mathbf{0}^T & -\tilde{\mathbf{x}}_i^T & y'_i \tilde{\mathbf{x}}_i^T \\
\tilde{\mathbf{x}}_i^T & \mathbf{0}^T & -x'_i \tilde{\mathbf{x}}_i^T
\end{pmatrix}
\mathbf{h} = \mathbf{0}
\end{aligned}
$$

where $\tilde{\mathbf{x}}_i = (x_i, y_i, 1)^T$. Stacking all $N$ correspondences:

$$
A \mathbf{h} = \mathbf{0}, \qquad A \in \mathbb{R}^{2N \times 9}
$$

### 3.4 SVD Solution

We seek the non-trivial $\mathbf{h}$ that minimizes $\|A\mathbf{h}\|^2$ subject to
$\|\mathbf{h}\| = 1$ (to avoid the trivial $\mathbf{h} = \mathbf{0}$).

Compute the SVD: $A = U \Sigma V^T$.

The solution is the **last column of $V$** (the right singular vector corresponding to
the smallest singular value $\sigma_9$).

**Why?** The SVD gives $\|A\mathbf{h}\|^2 = \|\Sigma V^T \mathbf{h}\|^2$. Setting
$\mathbf{y} = V^T \mathbf{h}$ (a rotation, so $\|\mathbf{y}\| = 1$):

$$
\|A\mathbf{h}\|^2 = \sum_{j=1}^{9} \sigma_j^2 y_j^2
$$

This is minimized when all weight is on the smallest $\sigma_j$: $\mathbf{y} = \mathbf{e}_9$,
giving $\mathbf{h} = V \mathbf{e}_9 = $ last column of $V$.

### 3.5 Hartley Normalization — Why It's Critical

**The problem**: Without normalization, the entries of $A$ span wildly different magnitudes.
If points have coordinates like $(500, 300, 1)$, then $A$ has entries ranging from
$O(1)$ to $O(10^5)$, leading to a poorly conditioned matrix. The SVD of a poorly
conditioned matrix amplifies floating-point errors.

**Hartley's normalization** (TPAMI 1997) applies a similarity transform to each point set
so that:
1. The centroid is at the origin
2. The mean distance from the origin is $\sqrt{2}$

$$
\bar{\mathbf{x}} = \frac{1}{N}\sum_{i} \mathbf{x}_i, \qquad
s = \frac{\sqrt{2}}{\frac{1}{N}\sum_{i} \|\mathbf{x}_i - \bar{\mathbf{x}}\|}
$$

The $3 \times 3$ normalization matrix is:

$$
T = \begin{pmatrix}
s & 0 & -s\bar{x} \\
0 & s & -s\bar{y} \\
0 & 0 & 1
\end{pmatrix}
$$

**Algorithm (Normalized DLT)**:
1. Compute $T_{\text{src}}$, $T_{\text{dst}}$ from source and destination points
2. Normalize: $\hat{\mathbf{x}}_i = T_{\text{src}} \tilde{\mathbf{x}}_i$, $\hat{\mathbf{x}}'_i = T_{\text{dst}} \tilde{\mathbf{x}}'_i$
3. Solve the DLT on normalized points → $\hat{H}$
4. Denormalize: $H = T_{\text{dst}}^{-1} \hat{H} T_{\text{src}}$

**Why denormalization works**: If $\hat{\mathbf{x}}' \sim \hat{H} \hat{\mathbf{x}}$, then
$T_{\text{dst}} \tilde{\mathbf{x}}' \sim \hat{H} T_{\text{src}} \tilde{\mathbf{x}}$,
so $\tilde{\mathbf{x}}' \sim T_{\text{dst}}^{-1} \hat{H} T_{\text{src}} \tilde{\mathbf{x}}$.

In [ ]:
def hartley_normalize(points):
    """Isotropic normalization: centroid to origin, mean distance = sqrt(2)."""
    points = np.asarray(points, dtype=np.float64)
    centroid = points.mean(axis=0)
    shifted = points - centroid
    mean_dist = np.sqrt((shifted ** 2).sum(axis=1)).mean()
    if mean_dist < 1e-12:
        raise ValueError("All points coincident.")
    scale = np.sqrt(2.0) / mean_dist
    T = np.array([
        [scale, 0.0, -scale * centroid[0]],
        [0.0, scale, -scale * centroid[1]],
        [0.0, 0.0, 1.0],
    ])
    pts_h = to_homogeneous(points)
    normalized_h = (T @ pts_h.T).T
    normalized = from_homogeneous(normalized_h)
    return normalized, T


def compute_homography_dlt(src_pts, dst_pts):
    """Estimate homography via normalized DLT (Algorithm 4.2 in H&Z)."""
    src_pts = np.asarray(src_pts, dtype=np.float64)
    dst_pts = np.asarray(dst_pts, dtype=np.float64)
    N = src_pts.shape[0]
    if N < 4:
        raise ValueError("Need >= 4 correspondences.")

    # Step 1: normalize
    src_norm, T_src = hartley_normalize(src_pts)
    dst_norm, T_dst = hartley_normalize(dst_pts)

    # Step 2: build the 2N x 9 matrix A
    A = np.zeros((2 * N, 9), dtype=np.float64)
    for i in range(N):
        x, y = src_norm[i]
        xp, yp = dst_norm[i]
        A[2 * i]     = [0, 0, 0, -x, -y, -1, yp*x, yp*y, yp]
        A[2 * i + 1] = [x, y, 1,  0,  0,  0, -xp*x, -xp*y, -xp]

    # Step 3: SVD
    _, S, Vt = np.linalg.svd(A)
    h = Vt[-1]  # last row of Vt = last column of V
    H_norm = h.reshape(3, 3)

    # Step 4: denormalize
    H = np.linalg.inv(T_dst) @ H_norm @ T_src
    H /= H[2, 2]
    return H, S

In [ ]:
# --- Demonstrate DLT with a known homography ---

# Ground-truth homography
H_true = np.array([
    [1.2, 0.3, -50],
    [0.1, 1.1,  20],
    [0.0003, 0.0005, 1.0]
])

np.random.seed(42)
src = np.random.uniform(50, 450, (20, 2))
dst = apply_homography_to_points(H_true, src)

H_est, singular_values = compute_homography_dlt(src, dst)

print("Ground-truth H:")
print(H_true / H_true[2, 2])
print("\nEstimated H (DLT):")
print(H_est)
print(f"\nMax absolute error: {np.max(np.abs(H_true / H_true[2,2] - H_est)):.2e}")
print(f"\nSingular values: {singular_values}")
print(f"Condition number (sigma_1/sigma_8): {singular_values[0]/singular_values[-2]:.2f}")
print(f"Smallest SV (should be ~0 for exact data): {singular_values[-1]:.2e}")

In [ ]:
# --- Condition number analysis: normalized vs unnormalized ---

def compute_homography_dlt_unnormalized(src_pts, dst_pts):
    """DLT WITHOUT Hartley normalization (for comparison)."""
    src_pts = np.asarray(src_pts, dtype=np.float64)
    dst_pts = np.asarray(dst_pts, dtype=np.float64)
    N = src_pts.shape[0]
    A = np.zeros((2 * N, 9), dtype=np.float64)
    for i in range(N):
        x, y = src_pts[i]
        xp, yp = dst_pts[i]
        A[2 * i]     = [0, 0, 0, -x, -y, -1, yp*x, yp*y, yp]
        A[2 * i + 1] = [x, y, 1,  0,  0,  0, -xp*x, -xp*y, -xp]
    _, S, Vt = np.linalg.svd(A)
    H = Vt[-1].reshape(3, 3)
    H /= H[2, 2]
    return H, S


noise_levels = [0.0, 0.1, 0.5, 1.0, 2.0, 5.0]
errors_norm = []
errors_unnorm = []
cond_norm = []
cond_unnorm = []

for noise in noise_levels:
    np.random.seed(123)
    src_pts = np.random.uniform(100, 900, (30, 2))
    dst_pts = apply_homography_to_points(H_true, src_pts)
    dst_noisy = dst_pts + np.random.randn(*dst_pts.shape) * noise

    H_n, S_n = compute_homography_dlt(src_pts, dst_noisy)
    H_u, S_u = compute_homography_dlt_unnormalized(src_pts, dst_noisy)

    proj_n = apply_homography_to_points(H_n, src_pts)
    proj_u = apply_homography_to_points(H_u, src_pts)
    errors_norm.append(np.mean(np.linalg.norm(proj_n - dst_pts, axis=1)))
    errors_unnorm.append(np.mean(np.linalg.norm(proj_u - dst_pts, axis=1)))
    cond_norm.append(S_n[0] / S_n[-2])
    cond_unnorm.append(S_u[0] / S_u[-2])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.semilogy(noise_levels, errors_norm, "o-", label="Normalized DLT", linewidth=2)
ax.semilogy(noise_levels, errors_unnorm, "s--", label="Unnormalized DLT", linewidth=2)
ax.set_xlabel("Noise σ (pixels)", fontsize=12)
ax.set_ylabel("Mean reprojection error (px)", fontsize=12)
ax.set_title("Reprojection Error: Normalized vs Unnormalized DLT", fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.semilogy(noise_levels, cond_norm, "o-", label="Normalized DLT", linewidth=2)
ax.semilogy(noise_levels, cond_unnorm, "s--", label="Unnormalized DLT", linewidth=2)
ax.set_xlabel("Noise σ (pixels)", fontsize=12)
ax.set_ylabel("Condition number σ₁/σ₈", fontsize=12)
ax.set_title("Condition Number of A Matrix", fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 4. RANSAC for Robust Estimation

### 4.1 The Outlier Problem

In practice, point correspondences come from feature matching (e.g., SIFT, ORB), which
inevitably produces **outliers** — incorrect matches. Even a single outlier can
catastrophically corrupt the DLT solution.

RANSAC (Random Sample Consensus, Fischler & Bolles 1981) is a **meta-algorithm** that
robustly estimates a model in the presence of outliers.

### 4.2 The Algorithm

```
for k = 1 to K:
    1. Randomly sample s=4 correspondences (minimal set for H)
    2. Compute H from the 4 points via DLT
    3. Count inliers: points where ||π(Hx̃ᵢ) - x'ᵢ|| < τ
    4. If more inliers than best so far, update best model
Refit H using ALL inliers of the best model
```

### 4.3 How Many Iterations? — Derivation

Let:
- $\varepsilon$ = outlier ratio (fraction of correspondences that are wrong)
- $s$ = sample size (4 for homography)
- $p$ = desired probability of finding a good model

The probability that a single sample of $s$ points is **all inliers**:

$$
q = (1 - \varepsilon)^s
$$

The probability that **all** $K$ samples contain at least one outlier:

$$
(1 - q)^K
$$

We want this failure probability to be at most $1 - p$:

$$
(1 - q)^K \le 1 - p
$$

Taking logarithms:

$$
K \ge \frac{\log(1 - p)}{\log(1 - (1 - \varepsilon)^s)}
$$

$$
\boxed{N = \left\lceil \frac{\log(1 - p)}{\log\left(1 - (1 - \varepsilon)^s\right)} \right\rceil}
$$

**Example**: With $\varepsilon = 0.5$ (50% outliers), $s = 4$, $p = 0.99$:

$$
N = \frac{\log(0.01)}{\log(1 - 0.5^4)} = \frac{\log(0.01)}{\log(0.9375)} = \frac{-4.605}{-0.0645} \approx 72
$$

In [ ]:
def ransac_iterations(outlier_ratio, sample_size=4, confidence=0.99):
    """Minimum RANSAC iterations for given outlier ratio and confidence."""
    q = (1 - outlier_ratio) ** sample_size
    if q < 1e-15:
        return float("inf")
    return int(np.ceil(np.log(1 - confidence) / np.log(1 - q)))


print("Required RANSAC iterations (p=0.99, s=4):")
print(f"{'Outlier %':>12}  {'Iterations':>12}")
print("-" * 28)
for eps in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    n = ransac_iterations(eps)
    print(f"{eps*100:>10.0f}%  {n:>12d}")

In [ ]:
def compute_homography_ransac(src_pts, dst_pts, thresh=3.0, max_iters=2000,
                               rng=None):
    """Robust homography estimation with RANSAC."""
    src_pts = np.asarray(src_pts, dtype=np.float64)
    dst_pts = np.asarray(dst_pts, dtype=np.float64)
    N = src_pts.shape[0]
    if N < 4:
        raise ValueError("Need >= 4 correspondences.")
    if rng is None:
        rng = np.random.default_rng()

    best_inlier_count = 0
    best_mask = np.zeros(N, dtype=bool)
    best_H = np.eye(3)
    src_h = to_homogeneous(src_pts)

    for _ in range(max_iters):
        idx = rng.choice(N, size=4, replace=False)
        try:
            H_cand, _ = compute_homography_dlt(src_pts[idx], dst_pts[idx])
        except (np.linalg.LinAlgError, ValueError):
            continue

        projected_h = (H_cand @ src_h.T).T
        w = projected_h[:, 2:3]
        valid = np.abs(w.ravel()) > 1e-12
        projected = np.full_like(dst_pts, np.inf)
        projected[valid] = projected_h[valid, :2] / w[valid]

        errors = np.linalg.norm(projected - dst_pts, axis=1)
        mask = errors < thresh
        n_in = mask.sum()

        if n_in > best_inlier_count:
            best_inlier_count = n_in
            best_mask = mask
            best_H = H_cand

    if best_inlier_count >= 4:
        best_H, _ = compute_homography_dlt(src_pts[best_mask], dst_pts[best_mask])

    return best_H, best_mask

In [ ]:
# --- Exercise: RANSAC with synthetic outliers ---

np.random.seed(42)
N_total = 100
N_outliers = 40  # 40% outliers

src_all = np.random.uniform(50, 450, (N_total, 2))
dst_all = apply_homography_to_points(H_true, src_all)
dst_all += np.random.randn(N_total, 2) * 1.0  # small inlier noise

outlier_idx = np.random.choice(N_total, N_outliers, replace=False)
dst_all[outlier_idx] = np.random.uniform(50, 500, (N_outliers, 2))

true_inlier_mask = np.ones(N_total, dtype=bool)
true_inlier_mask[outlier_idx] = False

H_dlt_all, _ = compute_homography_dlt(src_all, dst_all)
H_ransac, ransac_mask = compute_homography_ransac(
    src_all, dst_all, thresh=5.0, max_iters=2000,
    rng=np.random.default_rng(42)
)

proj_dlt = apply_homography_to_points(H_dlt_all, src_all)
proj_ransac = apply_homography_to_points(H_ransac, src_all)
dst_true = apply_homography_to_points(H_true, src_all)

err_dlt = np.mean(np.linalg.norm(proj_dlt - dst_true, axis=1))
err_ransac = np.mean(np.linalg.norm(proj_ransac - dst_true, axis=1))

print(f"Outlier ratio: {N_outliers/N_total:.0%}")
print(f"DLT (no RANSAC) — mean reprojection error: {err_dlt:.2f} px")
print(f"RANSAC DLT      — mean reprojection error: {err_ransac:.2f} px")
print(f"RANSAC inliers found: {ransac_mask.sum()}/{N_total}")
print(f"True inliers:         {true_inlier_mask.sum()}/{N_total}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, title, proj, err in [
    (axes[0], f"DLT (no RANSAC) — error: {err_dlt:.1f}px", proj_dlt, err_dlt),
    (axes[1], f"RANSAC DLT — error: {err_ransac:.1f}px", proj_ransac, err_ransac),
]:
    ax.scatter(dst_all[true_inlier_mask, 0], dst_all[true_inlier_mask, 1],
              c="green", s=30, label="Inliers", alpha=0.6)
    ax.scatter(dst_all[~true_inlier_mask, 0], dst_all[~true_inlier_mask, 1],
              c="red", s=30, label="Outliers", alpha=0.6)

    for i in range(N_total):
        ax.plot([dst_true[i, 0], proj[i, 0]], [dst_true[i, 1], proj[i, 1]],
                "b-", linewidth=0.5, alpha=0.3)

    ax.set_title(title, fontsize=12)
    ax.legend(fontsize=10)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)

plt.suptitle("RANSAC vs Plain DLT with 40% Outliers", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 5. Image Warping

### 5.1 Forward vs Inverse Warping

Given a homography $H$ mapping source to destination:

**Forward warping**: For each source pixel $(x, y)$, compute $(x', y') = \pi(H \tilde{\mathbf{x}})$
and copy the color to $(x', y')$ in the output.

**Problem**: $(x', y')$ is generally not at integer coordinates. Multiple source pixels may
map to the same output pixel, and some output pixels may receive no source pixel at all —
creating **holes**.

**Inverse warping**: For each output pixel $(x', y')$, compute the source location:

$$
\begin{pmatrix} x \\ y \\ w \end{pmatrix} = H^{-1} \begin{pmatrix} x' \\ y' \\ 1 \end{pmatrix},
\qquad (u, v) = (x/w, \; y/w)
$$

Then sample the source image at $(u, v)$ using interpolation. **Every output pixel gets a value**
— no holes!

### 5.2 Bilinear Interpolation

When $(u, v)$ falls between integer pixel positions, we interpolate from the four neighbours.
Let $u_0 = \lfloor u \rfloor$, $v_0 = \lfloor v \rfloor$, and define fractional parts
$a = u - u_0$, $b = v - v_0$:

$$
I(u, v) = (1-a)(1-b)\,I(u_0, v_0) + a(1-b)\,I(u_0+1, v_0) + (1-a)b\,I(u_0, v_0+1) + ab\,I(u_0+1, v_0+1)
$$

**Derivation**: This is simply two successive linear interpolations:
1. Interpolate horizontally: $I_{\text{top}} = (1-a) I_{00} + a I_{10}$, $I_{\text{bot}} = (1-a) I_{01} + a I_{11}$
2. Interpolate vertically: $I(u,v) = (1-b) I_{\text{top}} + b I_{\text{bot}}$

Expanding gives the formula above.

In [ ]:
def warp_image(image, H, output_shape):
    """Warp an image by homography H using inverse mapping + bilinear interpolation."""
    image = np.asarray(image)
    h_out, w_out = output_shape
    H_inv = np.linalg.inv(H)

    xs, ys = np.meshgrid(np.arange(w_out), np.arange(h_out))
    ones = np.ones_like(xs)
    dst_coords = np.stack([xs, ys, ones], axis=-1).reshape(-1, 3).T  # (3, N)

    src_coords = H_inv @ dst_coords
    src_coords /= src_coords[2:3, :]
    u = src_coords[0]
    v = src_coords[1]

    h_in, w_in = image.shape[:2]
    is_colour = image.ndim == 3

    u0 = np.floor(u).astype(np.int64)
    v0 = np.floor(v).astype(np.int64)
    u1, v1 = u0 + 1, v0 + 1

    a = (u - u0).astype(np.float64)
    b = (v - v0).astype(np.float64)

    valid = (u0 >= 0) & (v0 >= 0) & (u1 < w_in) & (v1 < h_in)

    u0c = np.clip(u0, 0, w_in - 1)
    v0c = np.clip(v0, 0, h_in - 1)
    u1c = np.clip(u1, 0, w_in - 1)
    v1c = np.clip(v1, 0, h_in - 1)

    if is_colour:
        I00 = image[v0c, u0c].astype(np.float64)
        I10 = image[v0c, u1c].astype(np.float64)
        I01 = image[v1c, u0c].astype(np.float64)
        I11 = image[v1c, u1c].astype(np.float64)
        a, b, valid = a[:, None], b[:, None], valid[:, None]
    else:
        I00 = image[v0c, u0c].astype(np.float64)
        I10 = image[v0c, u1c].astype(np.float64)
        I01 = image[v1c, u0c].astype(np.float64)
        I11 = image[v1c, u1c].astype(np.float64)

    interp = ((1-a)*(1-b)*I00 + a*(1-b)*I10 + (1-a)*b*I01 + a*b*I11)
    result = np.where(valid, interp, 0.0)

    if is_colour:
        result = result.reshape(h_out, w_out, image.shape[2])
    else:
        result = result.reshape(h_out, w_out)

    return np.clip(result, 0, 255).astype(np.uint8)

In [ ]:
# --- Demonstrate warping with synthetic scene ---

def make_synthetic_scene(size=300):
    """Create a colorful synthetic image with checkerboard + lines."""
    img = np.zeros((size, size, 3), dtype=np.uint8)

    sq = size // 6
    for r in range(6):
        for c in range(6):
            if (r + c) % 2 == 0:
                color = [200, 200, 200]
            else:
                color = [80, 80, 80]
            img[r*sq:(r+1)*sq, c*sq:(c+1)*sq] = color

    cv2.rectangle(img, (50, 50), (250, 250), (0, 120, 255), 3)
    cv2.circle(img, (150, 150), 60, (255, 100, 0), 3)
    cv2.line(img, (0, 150), (300, 150), (0, 200, 0), 2)
    cv2.line(img, (150, 0), (150, 300), (0, 200, 0), 2)
    cv2.putText(img, "P2", (120, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0,
                (255, 255, 0), 2)
    return img


scene = make_synthetic_scene(300)

H_perspective = np.array([
    [1.0,  0.3,  20],
    [0.2,  1.0,  10],
    [0.0008, 0.0005, 1.0]
])

warped_ours = warp_image(scene, H_perspective, (400, 500))
warped_cv2 = cv2.warpPerspective(scene, H_perspective, (500, 400))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(cv2.cvtColor(scene, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original Scene", fontsize=12)
axes[1].imshow(cv2.cvtColor(warped_ours, cv2.COLOR_BGR2RGB))
axes[1].set_title("Our warp_image() — Inverse + Bilinear", fontsize=12)
axes[2].imshow(cv2.cvtColor(warped_cv2, cv2.COLOR_BGR2RGB))
axes[2].set_title("cv2.warpPerspective (reference)", fontsize=12)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# --- Panorama stitching demo ---

def make_panorama_pair(base_size=300):
    """Create two overlapping synthetic views of a wide scene."""
    wide = np.zeros((base_size, base_size * 2, 3), dtype=np.uint8)
    sq = base_size // 6
    for r in range(6):
        for c in range(12):
            hue = int((c * 30 + r * 15) % 180)
            wide[r*sq:(r+1)*sq, c*sq:(c+1)*sq] = [
                100 + hue % 80, 120 + (hue * 2) % 100, 80 + (hue * 3) % 120
            ]

    for y in range(0, base_size, sq):
        cv2.line(wide, (0, y), (base_size * 2, y), (255, 255, 255), 1)
    for x in range(0, base_size * 2, sq):
        cv2.line(wide, (x, 0), (x, base_size), (255, 255, 255), 1)

    cv2.putText(wide, "LEFT", (80, 160), cv2.FONT_HERSHEY_SIMPLEX,
                1.5, (255, 0, 0), 3)
    cv2.putText(wide, "RIGHT", (380, 160), cv2.FONT_HERSHEY_SIMPLEX,
                1.5, (0, 0, 255), 3)

    overlap = int(base_size * 0.4)
    img1 = wide[:, :base_size + overlap // 2].copy()
    img2 = wide[:, base_size - overlap // 2:].copy()
    return img1, img2, wide


img1, img2, wide_truth = make_panorama_pair()

H_stitch = np.array([
    [1.0, 0.0, 240.0],
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0]
])

pano_h, pano_w = img1.shape[0], img1.shape[1] + img2.shape[1] - 120
canvas = np.zeros((pano_h, pano_w, 3), dtype=np.uint8)
canvas[:img1.shape[0], :img1.shape[1]] = img1
warped_2 = warp_image(img2, H_stitch, (pano_h, pano_w))

mask = warped_2.sum(axis=2) > 0
canvas[mask] = warped_2[mask]

fig, axes = plt.subplots(2, 2, figsize=(16, 8))
axes[0, 0].imshow(cv2.cvtColor(img1, cv2.COLOR_BGR2RGB))
axes[0, 0].set_title("View 1 (left)", fontsize=12)
axes[0, 1].imshow(cv2.cvtColor(img2, cv2.COLOR_BGR2RGB))
axes[0, 1].set_title("View 2 (right)", fontsize=12)
axes[1, 0].imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
axes[1, 0].set_title("Stitched Panorama (our warp_image)", fontsize=12)
axes[1, 1].imshow(cv2.cvtColor(wide_truth, cv2.COLOR_BGR2RGB))
axes[1, 1].set_title("Ground Truth Wide Scene", fontsize=12)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

---
## 6. Vanishing Points and Lines

### 6.1 Where Parallel Lines Meet

In the real world, parallel lines never meet. But a camera performs a projective
transformation, and under projection, parallel lines **converge at a vanishing point**.

**Derivation**: Consider two parallel 3D lines with direction $\mathbf{d}$:

$$
\mathbf{L}_1(t) = \mathbf{a}_1 + t\mathbf{d}, \qquad
\mathbf{L}_2(t) = \mathbf{a}_2 + t\mathbf{d}
$$

Under perspective projection with camera matrix $P = K[R | \mathbf{t}]$, a point at
parameter $t$ projects to:

$$
\tilde{\mathbf{x}}(t) = P \begin{pmatrix} \mathbf{a} + t\mathbf{d} \\ 1 \end{pmatrix}
= P \begin{pmatrix} \mathbf{a} \\ 1 \end{pmatrix} + t \, P \begin{pmatrix} \mathbf{d} \\ 0 \end{pmatrix}
$$

As $t \to \infty$, the projected point converges to:

$$
\mathbf{v} = P \begin{pmatrix} \mathbf{d} \\ 0 \end{pmatrix}
$$

This is the **vanishing point** — it depends only on the *direction* $\mathbf{d}$, not on the
specific lines.

### 6.2 In 2D: Lines as Cross Products

Given image line segments that are projections of parallel world lines:
1. Fit a line $\mathbf{l}_i = (a_i, b_i, c_i)$ to each segment
2. The vanishing point is $\mathbf{v} = \arg\min_{\|\mathbf{v}\|=1} \sum_i (\mathbf{l}_i^T \mathbf{v})^2$
3. This is solved by SVD: stack lines as rows of $L$, then $\mathbf{v}$ = last column of $V$ in $L = U\Sigma V^T$

### 6.3 Vanishing Line (Horizon)

Two sets of parallel lines on the same plane define two vanishing points $\mathbf{v}_1$,
$\mathbf{v}_2$. The **vanishing line** is:

$$
\mathbf{l}_\infty = \mathbf{v}_1 \times \mathbf{v}_2
$$

The vanishing line is the image of the plane's line at infinity — it's the **horizon** of
that plane.

### 6.4 Three Orthogonal Vanishing Points → Calibration

If we detect three mutually orthogonal vanishing points $\mathbf{v}_x, \mathbf{v}_y, \mathbf{v}_z$
(e.g., from a building's three edge directions), then the constraint

$$
\mathbf{v}_i^T \omega \mathbf{v}_j = 0 \quad \text{for } i \neq j
$$

where $\omega = (KK^T)^{-1}$ is the image of the absolute conic, gives constraints on
the calibration matrix $K$. Three orthogonal pairs give 3 constraints, enough to
determine $K$ with known principal point.

In [ ]:
def fit_line_from_endpoints(p1, p2):
    """Return homogeneous line (a,b,c) through two 2D points."""
    p1_h = to_homogeneous(np.asarray(p1, dtype=np.float64))
    p2_h = to_homogeneous(np.asarray(p2, dtype=np.float64))
    l = np.cross(p1_h, p2_h)
    n = np.linalg.norm(l[:2])
    if n > 1e-12:
        l /= n
    return l


def find_vanishing_point(lines):
    """Estimate vanishing point from near-parallel lines via SVD."""
    lines = np.asarray(lines, dtype=np.float64)
    _, _, Vt = np.linalg.svd(lines)
    vp = Vt[-1]
    return vp


def find_vanishing_line(vp1, vp2):
    """Vanishing line from two vanishing points."""
    vl = np.cross(vp1, vp2)
    n = np.linalg.norm(vl[:2])
    if n > 1e-12:
        vl /= n
    return vl

In [ ]:
# --- Synthetic architectural scene with vanishing points ---

def generate_architectural_scene():
    """Generate a synthetic scene with clear vanishing points.

    We create a simple building facade seen in perspective,
    with horizontal and vertical line segments.
    """
    img = np.ones((500, 700, 3), dtype=np.uint8) * 220

    vp_x = np.array([900.0, 250.0, 1.0])   # vanishing point for horizontal lines
    vp_y = np.array([350.0, -400.0, 1.0])   # vanishing point for vertical lines

    horizontal_lines = []
    vertical_lines = []

    base_left_pts = [(100, y) for y in range(80, 420, 40)]
    for x1, y1 in base_left_pts:
        dx = vp_x[0] - x1
        dy = vp_x[1] - y1
        length = 400
        t = length / np.sqrt(dx**2 + dy**2)
        x2 = int(x1 + t * dx)
        y2 = int(y1 + t * dy)
        x2 = min(x2, 650)
        cv2.line(img, (x1, y1), (x2, y2), (0, 0, 200), 2)
        horizontal_lines.append(fit_line_from_endpoints([x1, y1], [x2, y2]))

    base_top_pts = [(x, 80) for x in range(100, 600, 60)]
    for x1, y1 in base_top_pts:
        dx = vp_y[0] - x1
        dy = vp_y[1] - y1
        length = 380
        t = length / np.sqrt(dx**2 + dy**2)
        x2 = int(x1 + t * dx)
        y2 = int(y1 + t * dy)
        y2 = min(y2, 450)
        cv2.line(img, (x1, y1), (x2, y2), (200, 0, 0), 2)
        vertical_lines.append(fit_line_from_endpoints([x1, y1], [x2, y2]))

    return img, np.array(horizontal_lines), np.array(vertical_lines), vp_x, vp_y


scene_vp, h_lines, v_lines, vp_x_true, vp_y_true = generate_architectural_scene()

vp_x_est = find_vanishing_point(h_lines)
vp_y_est = find_vanishing_point(v_lines)

vp_x_e = from_homogeneous(vp_x_est)
vp_y_e = from_homogeneous(vp_y_est)
vp_x_true_e = from_homogeneous(vp_x_true)
vp_y_true_e = from_homogeneous(vp_y_true)

vl = find_vanishing_line(vp_x_est, vp_y_est)

print(f"Vanishing point X (horizontal lines):")
print(f"  True:      ({vp_x_true_e[0]:.1f}, {vp_x_true_e[1]:.1f})")
print(f"  Estimated: ({vp_x_e[0]:.1f}, {vp_x_e[1]:.1f})")
print(f"\nVanishing point Y (vertical lines):")
print(f"  True:      ({vp_y_true_e[0]:.1f}, {vp_y_true_e[1]:.1f})")
print(f"  Estimated: ({vp_y_e[0]:.1f}, {vp_y_e[1]:.1f})")
print(f"\nVanishing line: {vl[0]:.4f}x + {vl[1]:.4f}y + {vl[2]:.4f} = 0")

fig, ax = plt.subplots(1, 1, figsize=(12, 8))
ax.imshow(cv2.cvtColor(scene_vp, cv2.COLOR_BGR2RGB))

if 0 <= vp_x_e[0] <= 1200 and 0 <= vp_x_e[1] <= 800:
    ax.plot(*vp_x_e, "r*", markersize=20, zorder=10)
    ax.annotate(f"VP$_x$ ({vp_x_e[0]:.0f}, {vp_x_e[1]:.0f})",
                vp_x_e, textcoords="offset points", xytext=(10, 10),
                fontsize=12, color="red", fontweight="bold")

ax.annotate(f"VP$_y$ ({vp_y_e[0]:.0f}, {vp_y_e[1]:.0f}) [above image]",
            (350, 20), fontsize=12, color="blue", fontweight="bold",
            bbox=dict(boxstyle="round", fc="lightyellow", ec="blue"))

a, b, c = vl
if abs(b) > 1e-10:
    x_hl = np.array([0, 700])
    y_hl = -(a * x_hl + c) / b
    ax.plot(x_hl, y_hl, "g--", linewidth=2, label="Vanishing line (horizon)")

ax.set_title("Vanishing Point Detection from Line Segments", fontsize=14)
ax.legend(fontsize=11, loc="lower right")
plt.tight_layout()
plt.show()

---
## 7. Cross-Ratio Invariance

### 7.1 Definition

The **cross-ratio** of four collinear points $A, B, C, D$ is:

$$
\text{CR}(A, B, C, D) = \frac{|AC| \cdot |BD|}{|BC| \cdot |AD|}
$$

where $|PQ|$ denotes the **signed** distance along the line from $P$ to $Q$.

### 7.2 Proof of Invariance Under Projective Transformation

This is the **fundamental theorem** of projective invariants. We prove it using
homogeneous coordinates.

**Setup**: Let four collinear points on a line in $\mathbb{P}^1$ be represented in
homogeneous coordinates as $P_i = (\alpha_i, \beta_i)^T$ for $i = 1, 2, 3, 4$.

Since they are collinear, each can be written as $P_i = \mu_i \mathbf{e}_1 + \nu_i \mathbf{e}_2$
for some basis vectors $\mathbf{e}_1, \mathbf{e}_2$.

The signed distance ratio $|P_i P_j|$ is captured by the $2 \times 2$ determinant:

$$
\det(P_i, P_j) = \alpha_i \beta_j - \alpha_j \beta_i
$$

The cross-ratio in homogeneous form is:

$$
\text{CR}(P_1, P_2, P_3, P_4) = \frac{\det(P_1, P_3) \cdot \det(P_2, P_4)}{\det(P_2, P_3) \cdot \det(P_1, P_4)}
$$

**Under a projective transformation** $M \in GL(2)$ (a $2 \times 2$ invertible matrix),
the points transform as $P_i' = M P_i$.

Using the determinant identity for $2 \times 2$ matrices:

$$
\det(M P_i, M P_j) = \det(M) \cdot \det(P_i, P_j)
$$

**Proof of the identity**: Write $M = [\mathbf{m}_1 | \mathbf{m}_2]$ column-wise. Then
$MP_i = \alpha_i \mathbf{m}_1 + \beta_i \mathbf{m}_2$. The determinant of two such
vectors is:

$$
\det(MP_i, MP_j) = (\alpha_i \beta_j - \alpha_j \beta_i) \det(\mathbf{m}_1, \mathbf{m}_2)
= \det(P_i, P_j) \cdot \det(M)
$$

Now substituting into the cross-ratio:

$$
\text{CR}(P_1', P_2', P_3', P_4')
= \frac{\det(MP_1, MP_3) \cdot \det(MP_2, MP_4)}{\det(MP_2, MP_3) \cdot \det(MP_1, MP_4)}
= \frac{\det(M)^2 \cdot \det(P_1, P_3) \cdot \det(P_2, P_4)}
       {\det(M)^2 \cdot \det(P_2, P_3) \cdot \det(P_1, P_4)}
$$

The $\det(M)^2$ factors **cancel**:

$$
\boxed{\text{CR}(P_1', P_2', P_3', P_4') = \text{CR}(P_1, P_2, P_3, P_4)}
$$

$\square$

### 7.3 Connection to Single-View Metrology

Since cross-ratio is preserved under projection, if we can identify four collinear points
in an image whose cross-ratio has a known relationship to real-world measurements,
we can recover metric information from a single photograph. This is the foundation of
single-view metrology (Section 8).

In [ ]:
def cross_ratio(p1, p2, p3, p4):
    """Cross-ratio of four collinear points: CR = |AC|·|BD| / (|BC|·|AD|)."""
    p1 = np.asarray(p1, dtype=np.float64).ravel()
    p2 = np.asarray(p2, dtype=np.float64).ravel()
    p3 = np.asarray(p3, dtype=np.float64).ravel()
    p4 = np.asarray(p4, dtype=np.float64).ravel()

    d = p2 - p1
    norm = np.sqrt(np.dot(d, d))

    def signed_dist(a, b):
        return np.dot(b - a, d) / norm

    ac = signed_dist(p1, p3)
    bd = signed_dist(p2, p4)
    bc = signed_dist(p2, p3)
    ad = signed_dist(p1, p4)
    return (ac * bd) / (bc * ad)


# --- Demonstrate invariance ---
A = np.array([0.0, 0.0])
B = np.array([1.0, 0.0])
C = np.array([3.0, 0.0])
D = np.array([5.0, 0.0])

cr_original = cross_ratio(A, B, C, D)
print(f"Original points: A={A}, B={B}, C={C}, D={D}")
print(f"Cross-ratio: {cr_original:.6f}")

# Apply a projective transformation
H_test = np.array([
    [2.0, 0.5, 10],
    [0.3, 1.2, 5],
    [0.01, 0.005, 1.0]
])

pts = np.array([A, B, C, D])
pts_t = apply_homography_to_points(H_test, pts)
At, Bt, Ct, Dt = pts_t

cr_transformed = cross_ratio(At, Bt, Ct, Dt)
print(f"\nTransformed points:")
print(f"  A'={At}, B'={Bt}")
print(f"  C'={Ct}, D'={Dt}")
print(f"Cross-ratio after projective transform: {cr_transformed:.6f}")
print(f"\nDifference: {abs(cr_original - cr_transformed):.2e}")
print("Cross-ratio is INVARIANT under projective transformation!")

In [ ]:
# --- Visualize cross-ratio invariance ---

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax.set_title(f"Original — CR = {cr_original:.4f}", fontsize=13)
for pt, label in zip([A, B, C, D], ["A", "B", "C", "D"]):
    ax.plot(pt[0], pt[1], "ko", markersize=10)
    ax.annotate(label, pt, textcoords="offset points", xytext=(0, 12),
                fontsize=14, ha="center", fontweight="bold")
ax.plot([A[0]-0.5, D[0]+0.5], [0, 0], "b-", linewidth=2)
ax.set_ylim(-1, 1)
ax.set_aspect("equal")
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.set_title(f"After Projective Transform — CR = {cr_transformed:.4f}", fontsize=13)
for pt, label in zip([At, Bt, Ct, Dt], ["A'", "B'", "C'", "D'"]):
    ax.plot(pt[0], pt[1], "ro", markersize=10)
    ax.annotate(label, pt, textcoords="offset points", xytext=(0, 12),
                fontsize=14, ha="center", fontweight="bold")
ax.plot([At[0]-0.3, Dt[0]+0.3], [At[1]-0.3*(Dt[1]-At[1])/(Dt[0]-At[0]),
         Dt[1]+0.3*(Dt[1]-At[1])/(Dt[0]-At[0])], "r-", linewidth=2)
ax.set_aspect("equal")
ax.grid(True, alpha=0.3)

plt.suptitle("Cross-Ratio is Invariant Under Projective Transformation",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# --- Statistical verification: many random homographies ---

np.random.seed(0)
n_trials = 500
cr_diffs = []

for _ in range(n_trials):
    t_vals = sorted(np.random.uniform(-5, 5, 4))
    pts_orig = np.array([[t, 0.0] for t in t_vals])

    M = np.random.randn(3, 3) * 0.5
    M[2, 2] = 1.0
    M[2, :2] = np.random.randn(2) * 0.01
    if abs(np.linalg.det(M)) < 0.01:
        continue

    pts_proj = apply_homography_to_points(M, pts_orig)

    cr1 = cross_ratio(*pts_orig)
    cr2 = cross_ratio(*pts_proj)
    cr_diffs.append(abs(cr1 - cr2))

cr_diffs = np.array(cr_diffs)
print(f"Cross-ratio invariance test over {len(cr_diffs)} random homographies:")
print(f"  Max |ΔCR|:    {cr_diffs.max():.2e}")
print(f"  Mean |ΔCR|:   {cr_diffs.mean():.2e}")
print(f"  Median |ΔCR|: {np.median(cr_diffs):.2e}")
print("\nAll differences are at machine precision — invariance confirmed!")

---
## 8. Single-View Metrology

### 8.1 Measuring Heights from a Single Photograph

Given a single image of a scene with:
- A **reference object** of known height $H_r$ (e.g., a door = 2m)
- A **query object** whose height $H_q$ we want to measure
- The **vanishing point** $\mathbf{v}$ of vertical lines
- The **vanishing line** $\mathbf{l}_\infty$ of the ground plane

we can compute $H_q$ using the cross-ratio.

### 8.2 Derivation

Consider a vertical line through an object. Four special points on this line are:
1. $\mathbf{b}$ — the base (foot) of the object on the ground
2. $\mathbf{t}$ — the top of the object
3. $\mathbf{v}_h$ — the intersection of this vertical line with the horizon (vanishing line)
4. $\mathbf{v}$ — the vertical vanishing point

The cross-ratio of these four points is:

$$
\text{CR}(\mathbf{b}, \mathbf{t}, \mathbf{v}_h, \mathbf{v})
= \frac{|\mathbf{b}\mathbf{v}_h| \cdot |\mathbf{t}\mathbf{v}|}{|\mathbf{t}\mathbf{v}_h| \cdot |\mathbf{b}\mathbf{v}|}
$$

In the real world, $\mathbf{v}_h$ corresponds to the ground plane's point at infinity
(the horizon), and $\mathbf{v}$ is at infinity in the vertical direction. For a vertical
segment of height $H$:

$$
\text{CR} = \frac{H}{1} \cdot \frac{\infty}{\infty} \to H \text{ (after careful limiting)}
$$

Since the cross-ratio is invariant under projection, we compute the image cross-ratios
for the reference and query objects. In the 3D world, the four heights on the vertical
line are $(0,\; H,\; h_0,\; \infty)$, giving:

$$
\text{CR}(0, H; h_0, \infty) = \lim_{L\to\infty} \frac{(0-h_0)(H-L)}{(H-h_0)(0-L)} = \frac{h_0}{h_0 - H}
$$

where $h_0$ is the height at which the horizon plane cuts the vertical line. Solving
$H$ from the cross-ratio: $H = h_0 (1 - 1/\text{CR})$. Since $h_0$ is unknown but common
geometry, eliminating it via the reference object gives:

$$
\boxed{H_q = H_r \cdot \frac{\text{CR}_r\,(\text{CR}_q - 1)}{(\text{CR}_r - 1)\,\text{CR}_q}}
$$

where each cross-ratio is computed in the **image** using homogeneous coordinates:

$$
\text{CR}(\mathbf{b}, \mathbf{t}; \mathbf{v}_h, \mathbf{v})
= \frac{\det(\mathbf{b}, \mathbf{v}_h)\;\det(\mathbf{t}, \mathbf{v})}
       {\det(\mathbf{t}, \mathbf{v}_h)\;\det(\mathbf{b}, \mathbf{v})}
$$

with $\det(\mathbf{a}, \mathbf{b}) = (\mathbf{a} \times \mathbf{b})_z$ (the $z$-component
of the cross product of homogeneous 2D points, preserving sign). Here
$\mathbf{v}_{h,r}$ and $\mathbf{v}_{h,q}$ are the horizon intercepts of the vertical
lines through the reference and query objects respectively.

> **Warning:** Using unsigned Euclidean distances instead of signed projective
> determinants gives 3–13% error. The formula above is exact.

In [ ]:
def measure_height_single_view(base_point, top_point, reference_height,
                                ref_base, ref_top, vanishing_line, vertical_vp):
    """Estimate object height from a single view using cross-ratio."""
    def to_h(p):
        p = np.asarray(p, dtype=np.float64).ravel()
        return np.append(p, 1.0) if p.shape[0] == 2 else p.copy()

    def to_e(p):
        return p[:2] / p[2] if abs(p[2]) > 1e-12 else p[:2]

    b = to_h(base_point)
    t = to_h(top_point)
    br = to_h(ref_base)
    tr = to_h(ref_top)
    vl = np.asarray(vanishing_line, dtype=np.float64)
    vv = np.asarray(vertical_vp, dtype=np.float64)

    line_ref = np.cross(br, tr)
    vh_ref = np.cross(line_ref, vl)

    line_q = np.cross(b, t)
    vh_q = np.cross(line_q, vl)

    b_e, t_e = to_e(b), to_e(t)
    br_e, tr_e = to_e(br), to_e(tr)
    vv_e = to_e(vv)
    vh_ref_e = to_e(vh_ref)
    vh_q_e = to_e(vh_q)

    def det_h(a, b):
        """Signed determinant for homogeneous 2D points (z-component of cross product)."""
        return np.cross(a, b)[2]

    def cross_ratio(p1, p2, p3, p4):
        """CR(p1,p2;p3,p4) = det(p1,p3)*det(p2,p4) / (det(p2,p3)*det(p1,p4))"""
        return (det_h(p1, p3) * det_h(p2, p4)) / (det_h(p2, p3) * det_h(p1, p4))

    cr_q = cross_ratio(b, t, vh_q, vv)
    cr_r = cross_ratio(br, tr, vh_ref, vv)

    height = reference_height * cr_r * (cr_q - 1) / ((cr_r - 1) * cr_q)
    return float(height)

In [ ]:
# --- Exercise: estimate building height from synthetic photograph ---

def create_building_scene():
    """Create a synthetic perspective view of a building with a door.

    Returns the image and the geometric parameters needed for metrology.
    """
    img = np.ones((600, 800, 3), dtype=np.uint8) * 200

    # Sky gradient
    for y in range(250):
        t = y / 250.0
        img[y, :] = [int(200 * (1-t) + 135 * t),
                     int(220 * (1-t) + 206 * t),
                     int(255 * (1-t) + 235 * t)]

    # Ground
    img[250:, :] = [140, 160, 130]

    vertical_vp = np.array([400.0, -1500.0, 1.0])
    vanishing_line = np.array([0.0, 1.0, -250.0])  # y = 250

    # Building footprint: base corners and perspective top corners
    bldg_base_l, bldg_base_r = np.array([200, 500]), np.array([550, 500])

    def project_up(base, height_px):
        """Project a base point upward toward the vertical vanishing point."""
        b = to_homogeneous(base)
        v = vertical_vp
        direction = v[:2] / v[2] - base
        direction = direction / np.linalg.norm(direction)
        top = base + direction * height_px
        return top

    # Real-world heights: door = 2m, building = 10m
    door_height_real = 2.0    # meters
    building_height_real = 10.0  # meters

    door_px_h = 60
    door_base = np.array([280.0, 500.0])

    vp_euc = vertical_vp[:2] / vertical_vp[2]
    D = np.linalg.norm(vp_euc - door_base)
    t_ref = door_px_h / D
    d_eff = door_height_real * (1.0 - t_ref) / t_ref
    t_bldg = building_height_real / (building_height_real + d_eff)
    building_px_h = t_bldg * D

    door_top = project_up(door_base, door_px_h)

    bldg_base = np.array([280.0, 500.0])
    bldg_top = project_up(bldg_base, building_px_h)

    # Draw building
    bldg_tl = project_up(bldg_base_l, building_px_h)
    bldg_tr = project_up(bldg_base_r, building_px_h)
    pts_bldg = np.array([bldg_base_l, bldg_base_r,
                         bldg_tr.astype(int), bldg_tl.astype(int)])
    cv2.fillPoly(img, [pts_bldg], (180, 180, 200))
    cv2.polylines(img, [pts_bldg], True, (80, 80, 100), 2)

    # Draw door
    door_w = 25
    door_rect = np.array([
        [door_base[0] - door_w, door_base[1]],
        [door_base[0] + door_w, door_base[1]],
        project_up(np.array([door_base[0] + door_w, door_base[1]]), door_px_h).astype(int),
        project_up(np.array([door_base[0] - door_w, door_base[1]]), door_px_h).astype(int),
    ], dtype=np.int32)
    cv2.fillPoly(img, [door_rect], (80, 60, 40))
    cv2.polylines(img, [door_rect], True, (40, 30, 20), 2)

    # Draw windows
    for wx in [320, 380, 440, 500]:
        for wy_frac in [0.3, 0.55, 0.8]:
            wy = int(500 - building_px_h * wy_frac)
            cv2.rectangle(img, (wx-12, wy-15), (wx+12, wy+15), (170, 200, 220), -1)
            cv2.rectangle(img, (wx-12, wy-15), (wx+12, wy+15), (100, 120, 140), 1)

    # Annotations
    cv2.arrowedLine(img, (240, int(door_base[1])), (240, int(door_top[1])),
                    (0, 0, 255), 2, tipLength=0.05)
    cv2.arrowedLine(img, (240, int(door_top[1])), (240, int(door_base[1])),
                    (0, 0, 255), 2, tipLength=0.05)
    cv2.putText(img, "2m (ref)", (180, int((door_base[1]+door_top[1])/2)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)

    cv2.arrowedLine(img, (580, int(bldg_base[1])), (580, int(bldg_top[1])),
                    (255, 0, 0), 2, tipLength=0.02)
    cv2.arrowedLine(img, (580, int(bldg_top[1])), (580, int(bldg_base[1])),
                    (255, 0, 0), 2, tipLength=0.02)
    cv2.putText(img, "H = ?", (590, int((bldg_base[1]+bldg_top[1])/2)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

    return (img, door_base, door_top, bldg_base, bldg_top,
            vanishing_line, vertical_vp, door_height_real, building_height_real)


(scene_bldg, door_base, door_top, bldg_base, bldg_top,
 van_line, vert_vp, door_h_real, bldg_h_real) = create_building_scene()

estimated_height = measure_height_single_view(
    base_point=bldg_base,
    top_point=bldg_top,
    reference_height=door_h_real,
    ref_base=door_base,
    ref_top=door_top,
    vanishing_line=van_line,
    vertical_vp=vert_vp,
)

print(f"Reference: door height = {door_h_real:.1f} m")
print(f"True building height:    {bldg_h_real:.1f} m")
print(f"Estimated building height: {estimated_height:.2f} m")
print(f"Error: {abs(estimated_height - bldg_h_real):.2f} m "
      f"({abs(estimated_height - bldg_h_real)/bldg_h_real*100:.1f}%)")

fig, ax = plt.subplots(1, 1, figsize=(12, 9))
ax.imshow(cv2.cvtColor(scene_bldg, cv2.COLOR_BGR2RGB))
ax.axhline(y=250, color="green", linestyle="--", linewidth=2, alpha=0.7,
           label="Vanishing line (horizon)")

ax.plot(*door_base, "rv", markersize=10)
ax.plot(*door_top, "r^", markersize=10)
ax.plot(*bldg_base, "bv", markersize=10)
ax.plot(*bldg_top, "b^", markersize=10)

ax.annotate(f"Estimated: {estimated_height:.1f}m\n(True: {bldg_h_real:.1f}m)",
            xy=(620, 350), fontsize=13, color="blue", fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.4", fc="lightyellow", ec="blue", alpha=0.9))

ax.set_title("Single-View Metrology: Estimating Building Height", fontsize=14)
ax.legend(fontsize=11, loc="upper left")
ax.axis("off")
plt.tight_layout()
plt.show()

---
## Summary

### Key Takeaways

1. **Homogeneous coordinates** unify all 2D transformations into matrix multiplications.
   Points at infinity $(x,y,0)$ naturally represent directions.

2. The **2D transformation hierarchy** (isometry → similarity → affine → projective)
   progressively breaks more geometric structure while gaining degrees of freedom.

3. **Homography estimation** via DLT reduces to solving $A\mathbf{h} = \mathbf{0}$
   by SVD. **Hartley normalization** is essential for numerical stability.

4. **RANSAC** makes estimation robust to outliers. The number of iterations scales as
   $O\left(\frac{1}{(1-\varepsilon)^s}\right)$.

5. **Inverse warping** + bilinear interpolation gives hole-free image transformations.

6. **Vanishing points** are where parallel lines meet in the image. Two vanishing points
   define a **vanishing line** (horizon).

7. The **cross-ratio** is the unique projective invariant — it is preserved under any
   homography.

8. **Single-view metrology** exploits cross-ratio invariance to measure real-world
   distances from a single photograph.

### What's Next

In Notebook 03, we'll move from 2D projective geometry to 3D: the **pinhole camera model**,
intrinsic/extrinsic parameters, and camera calibration.